# 01_data_preprocessing.ipynb
## PhD Thesis Master Data Preprocessing Notebook

### Purpose
We are building a unified preprocessing pipeline for 12 datasets used in the PhD workflow. These datasets include:

- time-series datasets
- multi-entity load datasets
- system-level energy datasets
- smart meter datasets
- building datasets
- reinforcement learning and process-control datasets
- graph/network datasets
- regression-based generation datasets

### Why we are doing this
We are performing preprocessing before any baseline modeling or advanced algorithms because:

- all downstream notebooks depend on clean and standardized data
- reproducibility is required for thesis and IEEE publication quality
- timestamps, missing values, units, and schema consistency must be handled first
- dataset-specific structure must be preserved while keeping global conventions consistent

### Global conventions we are following
We are applying the following standards wherever appropriate:

- dataset-wise preprocessing first
- snake_case column naming
- `timestamp_utc` for real time-based datasets
- hourly processed versions where applicable
- no scaling during preprocessing
- metadata, quality reports, and manifest generation
- train / validation / test split generation
- preservation of dataset-specific canonical structure

### Canonical dataset families
We are processing datasets according to four canonical families:

1. Time-series datasets
2. Graph datasets
3. RL / process-control datasets
4. Tabular regression datasets

### Datasets included
1. UCI Household Electric Power Consumption
2. Electricity Load Diagrams 2011–2014
3. Open Power System Data (Time Series)
4. Low Carbon London Smart Meter
5. Building Data Genome Project 2
6. Tennessee Eastman Process (Braatz)
7. MATPOWER IEEE 118-Bus
8. Building Data Genome Project (Original)
9. PJM Hourly Energy Consumption
10. Industrial Park Load Dataset
11. Appliances Energy Prediction
12. Combined Cycle Power Plant

### What this notebook does
This notebook will:

- mount Google Drive
- automatically locate the correct base dataset folder
- locate and extract all ZIP files
- load tabular, MAT, and MATPOWER files
- preprocess each dataset using its appropriate canonical structure
- save processed outputs to Drive
- generate metadata, quality reports, and a master manifest

### Output structure
The notebook will create a structure similar to:

/processed/
- dataset_01_...
- dataset_02_...
- ...
- dataset_12_...

/manifests/
- manifest.json
- manifest.csv
- summary_report.json

### Downstream notebooks using these outputs
The outputs of this notebook will be used in:

- 02_baseline_models.ipynb
- 03_NEO_Forecast_v1.ipynb
- 04_system_modeling.ipynb
- 05_baseline_optimization.ipynb
- 06_NEO_Opt.ipynb
- 07_NEO_Robust.ipynb
- 08_NEO_Graph.ipynb
- 09_NEO_RL.ipynb
- 10_NEO_Edge.ipynb
- 11_full_system_simulation.ipynb
- 12_reproducibility_package.ipynb

### Important note
We are not reading the Google Drive share link directly as a filesystem path. Instead, after mounting Drive, we automatically discover the correct folder by searching for the dataset ZIP filenames you provided.

### Execution note
Run the code cell below in one go. It will:

- discover the correct base folder
- preprocess all datasets
- save outputs and reports
- build the master manifest

In [1]:
# =========================================================
# 01_data_preprocessing.ipynb
# Full master preprocessing pipeline for 12 datasets
# Google Colab single-cell implementation
# =========================================================

# -------------------------
# 0. INSTALLS AND IMPORTS
# -------------------------
!pip -q install pyarrow openpyxl xlrd scipy networkx

import os
import re
import gc
import io
import json
import math
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import io as sio
from google.colab import drive

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
np.random.seed(42)

# -------------------------
# 1. MOUNT GOOGLE DRIVE
# -------------------------
drive.mount('/content/drive')

# -------------------------
# 2. DATASET ZIP FILENAMES
# -------------------------
EXPECTED_ZIP_FILES = [
    "01_uci_household_power_FULL_PACKAGE.zip",
    "01_uci_household_power_usable_files.zip",
    "02_uci_backup_package.zip",
    "02_uci_electricity_load_diagrams_FULL_PACKAGE.zip",
    "03_opsd_time_series_FULL_PACKAGE.zip",
    "04_smartmeter_FULL_PACKAGE.zip",
    "05_building_data_genome_2_FULL_PACKAGE.zip",
    "06_tep_braatz_FULL_PACKAGE.zip",
    "07_matpower_case118_FULL_PACKAGE.zip",
    "08_building_data_genome_FULL_PACKAGE.zip",
    "09_pjm_hourly_energy_FULL_PACKAGE.zip",
    "10_industrial_park_load_FULL_PACKAGE.zip",
    "11_appliances_energy_prediction_FULL_PACKAGE.zip",
    "12_combined_cycle_power_plant_FULL_PACKAGE.zip",
]

PRIMARY_DATASET_ZIPS = [
    "01_uci_household_power_FULL_PACKAGE.zip",
    "01_uci_household_power_usable_files.zip",
    "02_uci_electricity_load_diagrams_FULL_PACKAGE.zip",
    "03_opsd_time_series_FULL_PACKAGE.zip",
    "04_smartmeter_FULL_PACKAGE.zip",
    "05_building_data_genome_2_FULL_PACKAGE.zip",
    "06_tep_braatz_FULL_PACKAGE.zip",
    "07_matpower_case118_FULL_PACKAGE.zip",
    "08_building_data_genome_FULL_PACKAGE.zip",
    "09_pjm_hourly_energy_FULL_PACKAGE.zip",
    "10_industrial_park_load_FULL_PACKAGE.zip",
    "11_appliances_energy_prediction_FULL_PACKAGE.zip",
    "12_combined_cycle_power_plant_FULL_PACKAGE.zip",
]

# -------------------------
# 3. AUTO-DISCOVER BASE DIR
# -------------------------
def discover_base_dir_from_drive(expected_files: List[str]) -> Path:
    """
    We search MyDrive for the folder that contains the highest number of expected ZIP files.
    This avoids relying on the Google Drive share link directly.
    """
    mydrive = Path("/content/drive/MyDrive")
    best_dir = None
    best_score = -1

    # candidate directories containing zip files
    candidate_dirs = set()
    for zip_path in mydrive.rglob("*.zip"):
        candidate_dirs.add(zip_path.parent)

    for d in candidate_dirs:
        files_in_dir = {p.name for p in d.glob("*.zip")}
        score = sum(1 for f in expected_files if f in files_in_dir)
        if score > best_score:
            best_score = score
            best_dir = d

    if best_dir is None or best_score <= 0:
        raise FileNotFoundError(
            "Could not automatically locate the dataset folder in MyDrive. "
            "Please verify that the ZIP files are present in Google Drive."
        )

    return best_dir.parent if best_dir.name == "raw_zips" else best_dir

DISCOVERED_BASE = discover_base_dir_from_drive(EXPECTED_ZIP_FILES)

# If the discovered folder already is raw_zips parent, use it directly.
# Otherwise, if the discovered folder itself is the ZIP folder, treat it as RAW_DIR.
if (DISCOVERED_BASE / "raw_zips").exists():
    BASE_DIR = DISCOVERED_BASE
    RAW_DIR = BASE_DIR / "raw_zips"
else:
    BASE_DIR = DISCOVERED_BASE
    RAW_DIR = BASE_DIR

EXTRACT_DIR = BASE_DIR / "extracted"
PROCESSED_DIR = BASE_DIR / "processed"
MANIFEST_DIR = BASE_DIR / "manifests"
LOG_DIR = BASE_DIR / "logs"

for p in [RAW_DIR, EXTRACT_DIR, PROCESSED_DIR, MANIFEST_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

# -------------------------
# 4. DATASET CONFIGURATION
# -------------------------
DATASET_CONFIGS: List[Dict[str, Any]] = [
    {
        "dataset_no": "01",
        "short_name": "uci_household_power",
        "display_name": "UCI Household Electric Power Consumption",
        "canonical_type": "time_series",
        "raw_zip_patterns": [
            "01_uci_household_power_FULL_PACKAGE.zip",
            "01_uci_household_power_usable_files.zip",
        ],
        "needs_hourly": True,
        "downstream_notebooks": ["02_baseline_models", "03_NEO_Forecast_v1", "10_NEO_Edge"],
        "target_columns_hint": ["global_active_power", "active_power_kw"],
    },
    {
        "dataset_no": "02",
        "short_name": "electricity_load_diagrams",
        "display_name": "Electricity Load Diagrams 2011–2014",
        "canonical_type": "time_series_multi_entity",
        "raw_zip_patterns": ["02_uci_electricity_load_diagrams_FULL_PACKAGE.zip"],
        "needs_hourly": True,
        "downstream_notebooks": ["02_baseline_models", "03_NEO_Forecast_v1"],
        "target_columns_hint": ["load_kwh", "load_mw"],
    },
    {
        "dataset_no": "03",
        "short_name": "opsd_time_series",
        "display_name": "Open Power System Data (Time Series)",
        "canonical_type": "system_time_series",
        "raw_zip_patterns": ["03_opsd_time_series_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": [
            "02_baseline_models", "03_NEO_Forecast_v1", "04_system_modeling",
            "05_baseline_optimization", "06_NEO_Opt", "07_NEO_Robust", "11_full_system_simulation"
        ],
        "target_columns_hint": ["load_actual", "load_mw"],
    },
    {
        "dataset_no": "04",
        "short_name": "smartmeter_low_carbon_london",
        "display_name": "Low Carbon London Smart Meter",
        "canonical_type": "time_series_multi_entity",
        "raw_zip_patterns": ["04_smartmeter_FULL_PACKAGE.zip"],
        "needs_hourly": True,
        "downstream_notebooks": ["10_NEO_Edge"],
        "target_columns_hint": ["energy_kwh", "consumption"],
    },
    {
        "dataset_no": "05",
        "short_name": "building_data_genome_2",
        "display_name": "Building Data Genome Project 2",
        "canonical_type": "time_series_multi_entity_with_metadata",
        "raw_zip_patterns": ["05_building_data_genome_2_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["04_system_modeling", "06_NEO_Opt", "11_full_system_simulation"],
        "target_columns_hint": ["energy_kwh", "meter_reading"],
    },
    {
        "dataset_no": "06",
        "short_name": "tep_braatz",
        "display_name": "Tennessee Eastman Process (Braatz)",
        "canonical_type": "rl_process",
        "raw_zip_patterns": ["06_tep_braatz_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["09_NEO_RL"],
        "target_columns_hint": [],
    },
    {
        "dataset_no": "07",
        "short_name": "ieee118_matpower",
        "display_name": "MATPOWER IEEE 118-Bus",
        "canonical_type": "graph",
        "raw_zip_patterns": ["07_matpower_case118_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["08_NEO_Graph"],
        "target_columns_hint": [],
    },
    {
        "dataset_no": "08",
        "short_name": "building_data_genome_original",
        "display_name": "Building Data Genome Project (Original)",
        "canonical_type": "time_series_multi_entity",
        "raw_zip_patterns": ["08_building_data_genome_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["03_NEO_Forecast_v1"],
        "target_columns_hint": ["energy_kwh", "meter_reading"],
    },
    {
        "dataset_no": "09",
        "short_name": "pjm_hourly_energy",
        "display_name": "PJM Hourly Energy Consumption",
        "canonical_type": "time_series_multi_entity",
        "raw_zip_patterns": ["09_pjm_hourly_energy_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["02_baseline_models", "03_NEO_Forecast_v1", "07_NEO_Robust"],
        "target_columns_hint": ["load_mw", "mw"],
    },
    {
        "dataset_no": "10",
        "short_name": "industrial_park_load",
        "display_name": "Industrial Park Load Dataset",
        "canonical_type": "time_series_industrial",
        "raw_zip_patterns": ["10_industrial_park_load_FULL_PACKAGE.zip"],
        "needs_hourly": True,
        "downstream_notebooks": ["05_baseline_optimization", "06_NEO_Opt", "07_NEO_Robust", "11_full_system_simulation"],
        "target_columns_hint": ["load_kw", "load_mw", "power"],
    },
    {
        "dataset_no": "11",
        "short_name": "appliances_energy_prediction",
        "display_name": "Appliances Energy Prediction",
        "canonical_type": "time_series_multivariate_single_entity",
        "raw_zip_patterns": ["11_appliances_energy_prediction_FULL_PACKAGE.zip"],
        "needs_hourly": True,
        "downstream_notebooks": ["02_baseline_models", "03_NEO_Forecast_v1", "10_NEO_Edge"],
        "target_columns_hint": ["appliances", "energy_kwh"],
    },
    {
        "dataset_no": "12",
        "short_name": "combined_cycle_power_plant",
        "display_name": "Combined Cycle Power Plant",
        "canonical_type": "tabular_regression",
        "raw_zip_patterns": ["12_combined_cycle_power_plant_FULL_PACKAGE.zip"],
        "needs_hourly": False,
        "downstream_notebooks": ["02_baseline_models", "04_system_modeling", "05_baseline_optimization", "06_NEO_Opt", "11_full_system_simulation"],
        "target_columns_hint": ["power_output_mw", "pe"],
    },
]

# -------------------------
# 5. GENERIC HELPERS
# -------------------------
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def write_log(msg: str, log_file: Path) -> None:
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{now_utc_iso()}] {msg}\n")

def snake_case(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[^A-Za-z0-9]+", "_", text)
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_").lower()

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [snake_case(c) for c in df.columns]
    return df

def save_json(obj: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def infer_time_column(cols: List[str]) -> Optional[str]:
    candidates = ["timestamp", "datetime", "date_time", "utc_timestamp", "date", "time"]
    lower_cols = [c.lower() for c in cols]
    for cand in candidates:
        if cand in lower_cols:
            return cols[lower_cols.index(cand)]
    return None

def find_matching_zip_files(patterns: List[str], raw_dir: Path) -> List[Path]:
    found = []
    for pattern in patterns:
        p = raw_dir / pattern
        if p.exists():
            found.append(p)
    return sorted(list(set(found)))

def extract_zip_recursive(zip_path: Path, out_dir: Path, max_depth: int = 2) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

    if max_depth <= 0:
        return

    nested_zips = list(out_dir.rglob("*.zip"))
    for nested_zip in nested_zips:
        nested_out = nested_zip.with_suffix("")
        if nested_out.exists():
            continue
        try:
            nested_out.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(nested_zip, "r") as zf:
                zf.extractall(nested_out)
        except Exception:
            pass

def discover_usable_files(root_dir: Path) -> List[Path]:
    usable_exts = {".csv", ".txt", ".tsv", ".xlsx", ".xls", ".json", ".mat", ".m"}
    return sorted([p for p in root_dir.rglob("*") if p.is_file() and p.suffix.lower() in usable_exts])

# -------------------------
# 6. FILE LOADERS
# -------------------------
def read_delimited(path: Path) -> pd.DataFrame:
    seps = [None, ",", ";", "\t", "|"]
    last_err = None
    for sep in seps:
        try:
            if sep is None:
                return pd.read_csv(path, sep=None, engine="python", na_values=["?", "NA", "N/A", "null", "NULL", ""])
            return pd.read_csv(path, sep=sep, engine="python", na_values=["?", "NA", "N/A", "null", "NULL", ""])
        except Exception as e:
            last_err = e
    raise last_err

def load_file(path: Path) -> Any:
    ext = path.suffix.lower()

    if ext in [".csv", ".txt", ".tsv"]:
        return read_delimited(path)
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if isinstance(obj, list):
            return pd.DataFrame(obj)
        if isinstance(obj, dict):
            return pd.json_normalize(obj)
        return pd.DataFrame({"value": [obj]})
    if ext == ".mat":
        return sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    if ext == ".m":
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    return None

# -------------------------
# 7. COMMON CLEANING
# -------------------------
def safe_convert_objects(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            try:
                converted = pd.to_numeric(df[c], errors="ignore")
                df[c] = converted
            except Exception:
                pass
    return df

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df)
    df = safe_convert_objects(df)
    df = df.drop_duplicates().reset_index(drop=True)
    return df

def schema_report(df: pd.DataFrame) -> Dict[str, Any]:
    return {
        "row_count": int(df.shape[0]),
        "column_count": int(df.shape[1]),
        "columns": list(df.columns),
        "dtypes": {c: str(df[c].dtype) for c in df.columns},
        "missing_per_column": {c: int(df[c].isna().sum()) for c in df.columns},
        "duplicate_rows": int(df.duplicated().sum()),
    }

def outlier_summary(df: pd.DataFrame) -> Dict[str, Any]:
    result = {}
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for c in num_cols:
        s = df[c].dropna()
        if len(s) < 5:
            continue
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        cnt = int(((s < lower) | (s > upper)).sum())
        result[c] = {
            "lower_bound": float(lower),
            "upper_bound": float(upper),
            "outlier_count": cnt,
        }
    return result

# -------------------------
# 8. TIME HANDLING AND FEATURES
# -------------------------
def try_create_timestamp(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    meta = {
        "timestamp_created": False,
        "timestamp_source": None,
        "original_sampling_interval": None,
    }
    df = df.copy()
    cols = list(df.columns)

    if "date" in cols and "time" in cols:
        try:
            df["timestamp_utc"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce",
                dayfirst=True,
                utc=True,
            )
            meta["timestamp_created"] = True
            meta["timestamp_source"] = "date+time"
        except Exception:
            pass
    else:
        ts_col = infer_time_column(cols)
        if ts_col is not None:
            try:
                df["timestamp_utc"] = pd.to_datetime(df[ts_col], errors="coerce", utc=True)
                meta["timestamp_created"] = True
                meta["timestamp_source"] = ts_col
            except Exception:
                pass

    if "timestamp_utc" in df.columns:
        df = df.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)
        if len(df) >= 2:
            diffs = df["timestamp_utc"].diff().dropna()
            if len(diffs) > 0:
                try:
                    meta["original_sampling_interval"] = str(diffs.mode().iloc[0])
                except Exception:
                    pass

    return df, meta

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    if "timestamp_utc" not in df.columns:
        return df
    df = df.copy()
    ts = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek
    df["month"] = ts.dt.month
    df["day_of_month"] = ts.dt.day
    df["is_weekend"] = ts.dt.dayofweek.isin([5, 6]).astype(int)
    df["quarter"] = ts.dt.quarter
    return df

def choose_target_column(df: pd.DataFrame, hints: List[str]) -> Optional[str]:
    cols = set(df.columns)
    for h in hints:
        if h in cols:
            return h
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return num_cols[0] if num_cols else None

def add_lag_rolling(df: pd.DataFrame, target_col: Optional[str], entity_col: Optional[str] = None) -> pd.DataFrame:
    if target_col is None or target_col not in df.columns or "timestamp_utc" not in df.columns:
        return df

    lag_steps = [1, 24, 168]
    rolling_steps = [3, 24, 168]
    df = df.copy()

    if entity_col and entity_col in df.columns:
        parts = []
        for _, part in df.groupby(entity_col):
            part = part.sort_values("timestamp_utc").copy()
            for lag in lag_steps:
                part[f"{target_col}_lag_{lag}"] = part[target_col].shift(lag)
            for win in rolling_steps:
                part[f"{target_col}_rolling_mean_{win}"] = part[target_col].rolling(win, min_periods=1).mean()
                part[f"{target_col}_rolling_std_{win}"] = part[target_col].rolling(win, min_periods=1).std()
            parts.append(part)
        return pd.concat(parts, ignore_index=True)

    df = df.sort_values("timestamp_utc").copy()
    for lag in lag_steps:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)
    for win in rolling_steps:
        df[f"{target_col}_rolling_mean_{win}"] = df[target_col].rolling(win, min_periods=1).mean()
        df[f"{target_col}_rolling_std_{win}"] = df[target_col].rolling(win, min_periods=1).std()
    return df

def resample_hourly(df: pd.DataFrame, entity_col: Optional[str] = None) -> pd.DataFrame:
    if "timestamp_utc" not in df.columns:
        return df.copy()

    df = df.copy()
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != entity_col]

    if entity_col and entity_col in df.columns:
        out = []
        for entity, part in df.groupby(entity_col):
            part = part.sort_values("timestamp_utc").set_index("timestamp_utc")
            rs = part[numeric_cols].resample("1H").mean().reset_index()
            rs[entity_col] = entity
            out.append(rs)
        return pd.concat(out, ignore_index=True) if out else df.copy()

    return (
        df.sort_values("timestamp_utc")
          .set_index("timestamp_utc")[numeric_cols]
          .resample("1H")
          .mean()
          .reset_index()
    )

def assign_split(df: pd.DataFrame, time_col: Optional[str] = "timestamp_utc") -> pd.DataFrame:
    df = df.copy()
    if time_col is not None and time_col in df.columns:
        df = df.sort_values(time_col).reset_index(drop=True)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    split = np.array(["train"] * n)
    split[train_end:val_end] = "validation"
    split[val_end:] = "test"
    df["split"] = split
    return df

# -------------------------
# 9. SPECIAL STRUCTURE HELPERS
# -------------------------
def wide_to_long(df: pd.DataFrame, timestamp_col: str) -> pd.DataFrame:
    value_vars = [c for c in df.columns if c != timestamp_col]
    long_df = df.melt(
        id_vars=[timestamp_col],
        value_vars=value_vars,
        var_name="entity_id",
        value_name="target_value",
    )
    return long_df

def parse_matpower_case_text(case_text: str) -> Dict[str, pd.DataFrame]:
    result = {}
    patterns = {
        "bus": r"mpc\.bus\s*=\s*\[(.*?)\];",
        "branch": r"mpc\.branch\s*=\s*\[(.*?)\];",
        "gen": r"mpc\.gen\s*=\s*\[(.*?)\];",
    }
    for key, pattern in patterns.items():
        match = re.search(pattern, case_text, flags=re.S)
        if match:
            block = match.group(1).strip()
            rows = []
            for line in block.split("\n"):
                line = line.split("%")[0].strip().rstrip(";")
                if not line:
                    continue
                vals = [v for v in re.split(r"\s+", line) if v]
                try:
                    rows.append([float(v) for v in vals])
                except Exception:
                    pass
            if rows:
                result[key] = pd.DataFrame(rows)
    return result

def extract_rl_tables(mat_dict: Dict[str, Any]) -> Dict[str, pd.DataFrame]:
    outputs = {}
    for k, v in mat_dict.items():
        if k.startswith("__"):
            continue
        if isinstance(v, np.ndarray):
            arr = np.atleast_2d(v)
            if arr.ndim == 2:
                outputs[k] = pd.DataFrame(arr)
    return outputs

# -------------------------
# 10. TIME-SERIES PROCESSOR
# -------------------------
def process_time_series_dataset(cfg: Dict[str, Any], files: List[Path], out_dir: Path, log_path: Path, manifest_entry: Dict[str, Any]) -> Dict[str, Any]:
    tabular = []

    for path in files:
        try:
            obj = load_file(path)
            if isinstance(obj, pd.DataFrame) and len(obj) > 0:
                df = clean_dataframe(obj)
                tabular.append((path, df))
        except Exception as e:
            write_log(f"Failed to load {path.name}: {e}", log_path)

    if not tabular:
        manifest_entry["status"] = "no_tabular_files_loaded"
        manifest_entry["finished_at_utc"] = now_utc_iso()
        return manifest_entry

    primary_path, df = sorted(tabular, key=lambda x: x[1].shape[0] * x[1].shape[1], reverse=True)[0]
    write_log(f"Primary file selected: {primary_path.name}", log_path)

    pre_schema = schema_report(df)
    df, ts_meta = try_create_timestamp(df)

    entity_col = None
    if cfg["canonical_type"] in ["time_series_multi_entity", "time_series_multi_entity_with_metadata"]:
        if "timestamp_utc" in df.columns:
            special_id_cols = ["entity_id", "building_id", "region_id", "household_id", "meter_id"]
            existing = [c for c in special_id_cols if c in df.columns]
            if existing:
                entity_col = existing[0]
            else:
                non_ts_cols = [c for c in df.columns if c != "timestamp_utc"]
                if len(non_ts_cols) > 5:
                    df = wide_to_long(df, "timestamp_utc")
                    entity_col = "entity_id"

    # Harmonized renaming
    rename_map = {}
    for col in df.columns:
        if col == "global_active_power":
            rename_map[col] = "active_power_kw"
        elif col == "global_reactive_power":
            rename_map[col] = "reactive_power_kvar"
        elif col == "global_intensity":
            rename_map[col] = "current_a"
        elif col == "voltage":
            rename_map[col] = "voltage_v"
        elif col == "appliances":
            rename_map[col] = "energy_kwh"
        elif col == "pe":
            rename_map[col] = "power_output_mw"
    df = df.rename(columns=rename_map)

    target_col = choose_target_column(df, cfg["target_columns_hint"])

    df = add_time_features(df)
    df = add_lag_rolling(df, target_col=target_col, entity_col=entity_col)
    df = assign_split(df, time_col="timestamp_utc" if "timestamp_utc" in df.columns else None)

    raw_clean_path = out_dir / f"{cfg['short_name']}_clean_raw.parquet"
    df.to_parquet(raw_clean_path, index=False)

    manifest_entry["processed_outputs"].append(str(raw_clean_path))

    hourly_path = None
    if cfg["needs_hourly"] and "timestamp_utc" in df.columns:
        hourly_df = resample_hourly(df, entity_col=entity_col)
        hourly_df = add_time_features(hourly_df)
        hourly_target = choose_target_column(hourly_df, cfg["target_columns_hint"])
        hourly_df = add_lag_rolling(hourly_df, target_col=hourly_target, entity_col=entity_col)
        hourly_df = assign_split(hourly_df, time_col="timestamp_utc")
        hourly_path = out_dir / f"{cfg['short_name']}_clean_hourly.parquet"
        hourly_df.to_parquet(hourly_path, index=False)
        manifest_entry["processed_outputs"].append(str(hourly_path))

    quality = {
        "schema_before": pre_schema,
        "schema_after": schema_report(df),
        "timestamp_metadata": ts_meta,
        "outlier_summary": outlier_summary(df),
        "canonical_type": cfg["canonical_type"],
        "entity_column": entity_col,
        "target_column": target_col,
        "source_file": str(primary_path),
        "raw_clean_path": str(raw_clean_path),
        "hourly_path": str(hourly_path) if hourly_path else None,
    }

    save_json(quality, out_dir / "quality_report.json")
    save_json({"columns": list(df.columns)}, out_dir / "feature_manifest.json")

    manifest_entry["processed_outputs"].append(str(out_dir / "quality_report.json"))
    manifest_entry["processed_outputs"].append(str(out_dir / "feature_manifest.json"))
    manifest_entry["status"] = "completed"
    manifest_entry["finished_at_utc"] = now_utc_iso()
    manifest_entry["row_count"] = int(df.shape[0])
    manifest_entry["column_count"] = int(df.shape[1])
    manifest_entry["target_column"] = target_col
    return manifest_entry

# -------------------------
# 11. GRAPH PROCESSOR
# -------------------------
def process_graph_dataset(cfg: Dict[str, Any], files: List[Path], out_dir: Path, log_path: Path, manifest_entry: Dict[str, Any]) -> Dict[str, Any]:
    m_files = [p for p in files if p.suffix.lower() == ".m"]
    if not m_files:
        manifest_entry["status"] = "graph_source_not_found"
        manifest_entry["finished_at_utc"] = now_utc_iso()
        return manifest_entry

    source_path = m_files[0]
    case_text = load_file(source_path)
    parsed = parse_matpower_case_text(case_text)

    node_count = 0
    edge_count = 0

    if "bus" in parsed:
        nodes = parsed["bus"].copy()
        nodes.columns = [f"feature_{i}" for i in range(nodes.shape[1])]
        if nodes.shape[1] > 0:
            nodes = nodes.rename(columns={"feature_0": "bus_id"})
        nodes.to_csv(out_dir / "nodes.csv", index=False)
        manifest_entry["processed_outputs"].append(str(out_dir / "nodes.csv"))
        node_count = len(nodes)

    if "branch" in parsed:
        edges = parsed["branch"].copy()
        edges.columns = [f"feature_{i}" for i in range(edges.shape[1])]
        if edges.shape[1] > 1:
            edges = edges.rename(columns={"feature_0": "from_bus", "feature_1": "to_bus"})
        edges.to_csv(out_dir / "edges.csv", index=False)
        manifest_entry["processed_outputs"].append(str(out_dir / "edges.csv"))
        edge_count = len(edges)

    if "gen" in parsed:
        gen = parsed["gen"].copy()
        gen.columns = [f"feature_{i}" for i in range(gen.shape[1])]
        gen.to_csv(out_dir / "generators.csv", index=False)
        manifest_entry["processed_outputs"].append(str(out_dir / "generators.csv"))

    save_json({
        "canonical_type": "graph",
        "source_file": str(source_path),
        "node_count": node_count,
        "edge_count": edge_count,
        "created_at_utc": now_utc_iso(),
    }, out_dir / "graph_metadata.json")

    manifest_entry["processed_outputs"].append(str(out_dir / "graph_metadata.json"))
    manifest_entry["status"] = "completed"
    manifest_entry["finished_at_utc"] = now_utc_iso()
    return manifest_entry

# -------------------------
# 12. RL / PROCESS PROCESSOR
# -------------------------
def process_rl_dataset(cfg: Dict[str, Any], files: List[Path], out_dir: Path, log_path: Path, manifest_entry: Dict[str, Any]) -> Dict[str, Any]:
    tables = {}

    for path in files:
        try:
            obj = load_file(path)
            if isinstance(obj, dict):
                extracted = extract_rl_tables(obj)
                for k, v in extracted.items():
                    tables[f"{path.stem}_{k}"] = v
            elif isinstance(obj, pd.DataFrame):
                tables[path.stem] = clean_dataframe(obj)
        except Exception as e:
            write_log(f"Failed RL load {path.name}: {e}", log_path)

    if not tables:
        manifest_entry["status"] = "no_rl_tables_loaded"
        manifest_entry["finished_at_utc"] = now_utc_iso()
        return manifest_entry

    ordered = sorted(tables.items(), key=lambda kv: kv[1].shape[0] * max(1, kv[1].shape[1]), reverse=True)

    states_df = ordered[0][1].copy()
    states_df.columns = [f"state_{i}" for i in range(states_df.shape[1])]
    states_df.insert(0, "time_step", np.arange(len(states_df)))
    states_df.to_csv(out_dir / "states.csv", index=False)
    manifest_entry["processed_outputs"].append(str(out_dir / "states.csv"))

    if len(ordered) > 1:
        actions_df = ordered[1][1].copy()
        actions_df.columns = [f"action_{i}" for i in range(actions_df.shape[1])]
        actions_df.insert(0, "time_step", np.arange(len(actions_df)))
    else:
        actions_df = pd.DataFrame({"time_step": np.arange(len(states_df))})

    actions_df.to_csv(out_dir / "actions.csv", index=False)
    manifest_entry["processed_outputs"].append(str(out_dir / "actions.csv"))

    episodes_df = pd.DataFrame({
        "episode_id": [0],
        "start_index": [0],
        "end_index": [max(0, len(states_df) - 1)],
    })
    episodes_df.to_csv(out_dir / "episodes.csv", index=False)
    manifest_entry["processed_outputs"].append(str(out_dir / "episodes.csv"))

    save_json({
        "canonical_type": "rl_process",
        "state_rows": int(states_df.shape[0]),
        "state_cols": int(states_df.shape[1]),
        "action_rows": int(actions_df.shape[0]),
        "action_cols": int(actions_df.shape[1]),
        "created_at_utc": now_utc_iso(),
    }, out_dir / "rl_metadata.json")

    manifest_entry["processed_outputs"].append(str(out_dir / "rl_metadata.json"))
    manifest_entry["status"] = "completed"
    manifest_entry["finished_at_utc"] = now_utc_iso()
    return manifest_entry

# -------------------------
# 13. TABULAR REGRESSION PROCESSOR
# -------------------------
def process_tabular_regression_dataset(cfg: Dict[str, Any], files: List[Path], out_dir: Path, log_path: Path, manifest_entry: Dict[str, Any]) -> Dict[str, Any]:
    tabular = []

    for path in files:
        try:
            obj = load_file(path)
            if isinstance(obj, pd.DataFrame) and len(obj) > 0:
                tabular.append((path, clean_dataframe(obj)))
        except Exception as e:
            write_log(f"Failed regression load {path.name}: {e}", log_path)

    if not tabular:
        manifest_entry["status"] = "no_tabular_files_loaded"
        manifest_entry["finished_at_utc"] = now_utc_iso()
        return manifest_entry

    primary_path, df = sorted(tabular, key=lambda x: x[1].shape[0] * x[1].shape[1], reverse=True)[0]

    rename_map = {}
    for col in df.columns:
        c = col.lower()
        if c in ["t", "temperature"]:
            rename_map[col] = "temperature_c"
        elif c in ["ap", "ambient_pressure", "pressure"]:
            rename_map[col] = "pressure_bar"
        elif c in ["rh", "humidity"]:
            rename_map[col] = "humidity_percent"
        elif c in ["v", "exhaust_vacuum"]:
            rename_map[col] = "exhaust_vacuum"
        elif c in ["pe", "power_output", "power"]:
            rename_map[col] = "power_output_mw"

    df = df.rename(columns=rename_map)
    df = assign_split(df, time_col=None)

    clean_path = out_dir / f"{cfg['short_name']}_clean.parquet"
    df.to_parquet(clean_path, index=False)

    save_json({
        "schema": schema_report(df),
        "outlier_summary": outlier_summary(df),
        "canonical_type": "tabular_regression",
        "source_file": str(primary_path),
    }, out_dir / "quality_report.json")

    manifest_entry["processed_outputs"].append(str(clean_path))
    manifest_entry["processed_outputs"].append(str(out_dir / "quality_report.json"))
    manifest_entry["status"] = "completed"
    manifest_entry["finished_at_utc"] = now_utc_iso()
    manifest_entry["row_count"] = int(df.shape[0])
    manifest_entry["column_count"] = int(df.shape[1])
    manifest_entry["target_column"] = choose_target_column(df, cfg["target_columns_hint"])
    return manifest_entry

# -------------------------
# 14. MASTER DATASET PROCESSOR
# -------------------------
def process_dataset(cfg: Dict[str, Any]) -> Dict[str, Any]:
    dataset_no = cfg["dataset_no"]
    short_name = cfg["short_name"]
    dataset_id = f"dataset_{dataset_no}_{short_name}"

    extract_dir = EXTRACT_DIR / dataset_id
    out_dir = PROCESSED_DIR / dataset_id
    log_file = LOG_DIR / f"{dataset_id}.log"

    extract_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest_entry = {
        "dataset_no": dataset_no,
        "dataset_id": dataset_id,
        "display_name": cfg["display_name"],
        "canonical_type": cfg["canonical_type"],
        "downstream_notebooks": cfg["downstream_notebooks"],
        "processed_outputs": [],
        "status": "started",
        "started_at_utc": now_utc_iso(),
    }

    raw_zip_files = find_matching_zip_files(cfg["raw_zip_patterns"], RAW_DIR)
    manifest_entry["raw_zip_files"] = [str(p) for p in raw_zip_files]

    if not raw_zip_files:
        manifest_entry["status"] = "raw_zip_not_found"
        manifest_entry["finished_at_utc"] = now_utc_iso()
        return manifest_entry

    write_log(f"Starting {dataset_id}", log_file)

    for zp in raw_zip_files:
        try:
            extract_zip_recursive(zp, extract_dir)
            write_log(f"Extracted {zp.name}", log_file)
        except Exception as e:
            write_log(f"Failed extraction {zp.name}: {e}", log_file)

    files = discover_usable_files(extract_dir)
    manifest_entry["discovered_files"] = [str(p) for p in files]

    if cfg["canonical_type"] == "graph":
        return process_graph_dataset(cfg, files, out_dir, log_file, manifest_entry)
    elif cfg["canonical_type"] == "rl_process":
        return process_rl_dataset(cfg, files, out_dir, log_file, manifest_entry)
    elif cfg["canonical_type"] == "tabular_regression":
        return process_tabular_regression_dataset(cfg, files, out_dir, log_file, manifest_entry)
    else:
        return process_time_series_dataset(cfg, files, out_dir, log_file, manifest_entry)

# -------------------------
# 15. EXECUTION ACROSS ALL DATASETS
# -------------------------
master_manifest = []

for cfg in DATASET_CONFIGS:
    print(f"\nProcessing Dataset {cfg['dataset_no']}: {cfg['display_name']}")
    try:
        result = process_dataset(cfg)
    except Exception as e:
        result = {
            "dataset_no": cfg["dataset_no"],
            "dataset_id": f"dataset_{cfg['dataset_no']}_{cfg['short_name']}",
            "display_name": cfg["display_name"],
            "canonical_type": cfg["canonical_type"],
            "status": f"failed: {str(e)}",
            "finished_at_utc": now_utc_iso(),
            "processed_outputs": [],
        }
    master_manifest.append(result)
    gc.collect()

manifest_df = pd.DataFrame(master_manifest)
display_cols = [c for c in ["dataset_no", "display_name", "canonical_type", "status", "row_count", "column_count"] if c in manifest_df.columns]
display(manifest_df[display_cols])

# -------------------------
# 16. SAVE MASTER MANIFEST
# -------------------------
manifest_json_path = MANIFEST_DIR / "manifest.json"
manifest_csv_path = MANIFEST_DIR / "manifest.csv"
summary_json_path = MANIFEST_DIR / "summary_report.json"

save_json({
    "generated_at_utc": now_utc_iso(),
    "base_dir": str(BASE_DIR),
    "raw_dir": str(RAW_DIR),
    "datasets": master_manifest,
}, manifest_json_path)

manifest_df.to_csv(manifest_csv_path, index=False)

summary_report = {
    "generated_at_utc": now_utc_iso(),
    "base_dir": str(BASE_DIR),
    "raw_dir": str(RAW_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "total_datasets_configured": len(DATASET_CONFIGS),
    "total_datasets_completed": int((manifest_df["status"] == "completed").sum()) if "status" in manifest_df.columns else 0,
    "total_datasets_incomplete": int((manifest_df["status"] != "completed").sum()) if "status" in manifest_df.columns else 0,
    "manifest_json": str(manifest_json_path),
    "manifest_csv": str(manifest_csv_path),
}
save_json(summary_report, summary_json_path)

print("\nSaved:")
print("Manifest JSON:", manifest_json_path)
print("Manifest CSV :", manifest_csv_path)
print("Summary JSON :", summary_json_path)

# -------------------------
# 17. FINAL VALIDATION
# -------------------------
completed = manifest_df[manifest_df["status"] == "completed"] if "status" in manifest_df.columns else pd.DataFrame()
incomplete = manifest_df[manifest_df["status"] != "completed"] if "status" in manifest_df.columns else pd.DataFrame()

print("\nCompleted datasets:", len(completed))
print("Incomplete datasets:", len(incomplete))

if len(incomplete) > 0:
    cols = [c for c in ["dataset_no", "display_name", "status"] if c in incomplete.columns]
    display(incomplete[cols])

# -------------------------
# 18. OPTIONAL PREVIEW
# -------------------------
preview_paths = []
for row in master_manifest:
    for p in row.get("processed_outputs", []):
        if p.endswith(".parquet"):
            preview_paths.append(Path(p))
            break
    if len(preview_paths) >= 2:
        break

for p in preview_paths:
    print("\nPreviewing:", p.name)
    try:
        df_preview = pd.read_parquet(p)
        display(df_preview.head())
    except Exception as e:
        print("Could not preview:", e)

print("\n01_data_preprocessing.ipynb completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/REAL DATASETS
RAW_DIR: /content/drive/MyDrive/REAL DATASETS
EXTRACT_DIR: /content/drive/MyDrive/REAL DATASETS/extracted
PROCESSED_DIR: /content/drive/MyDrive/REAL DATASETS/processed

Processing Dataset 01: UCI Household Electric Power Consumption

Processing Dataset 02: Electricity Load Diagrams 2011–2014

Processing Dataset 03: Open Power System Data (Time Series)

Processing Dataset 04: Low Carbon London Smart Meter

Processing Dataset 05: Building Data Genome Project 2

Processing Dataset 06: Tennessee Eastman Process (Braatz)

Processing Dataset 07: MATPOWER IEEE 118-Bus

Processing Dataset 08: Building Data Genome Project (Original)

Processing Dataset 09: PJM Hourly Energy Consumption

Processing Dataset 10: Industrial Park Load Dataset

Processing Dataset 11: Appliances Energy Prediction

Processing Dataset 12: Combined

,dataset_no,display_name,canonical_type,status,row_count,column_count
0,01,UCI Household Electric Power Consumption,time_series,completed,2075259.0,26.0
1,02,Electricity Load Diagrams 2011–2014,time_series_multi_entity,completed,140256.0,372.0
2,03,Open Power System Data (Time Series),system_time_series,completed,50401.0,317.0
3,04,Low Carbon London Smart Meter,time_series_multi_entity,"failed: (""Expected bytes, got a 'float' object...",NaN,NaN
4,05,Building Data Genome Project 2,time_series_multi_entity_with_metadata,completed,2.0,3.0
5,06,Tennessee Eastman Process (Braatz),rl_process,completed,NaN,NaN
6,07,MATPOWER IEEE 118-Bus,graph,completed,NaN,NaN
7,08,Building Data Genome Project (Original),time_series_multi_entity,"failed: (""Expected bytes, got a 'float' object...",NaN,NaN
8,09,PJM Hourly Energy Consumption,time_series_multi_entity,completed,121273.0,19.0
9,10,Industrial Park Load Dataset,time_series_industrial,completed,80.0,3.0



Saved:
Manifest JSON: /content/drive/MyDrive/REAL DATASETS/manifests/manifest.json
Manifest CSV : /content/drive/MyDrive/REAL DATASETS/manifests/manifest.csv
Summary JSON : /content/drive/MyDrive/REAL DATASETS/manifests/summary_report.json

Completed datasets: 10
Incomplete datasets: 2


,dataset_no,display_name,status
3,04,Low Carbon London Smart Meter,"failed: (""Expected bytes, got a 'float' object..."
7,08,Building Data Genome Project (Original),"failed: (""Expected bytes, got a 'float' object..."



Previewing: uci_household_power_clean_raw.parquet


,date,time,active_power_kw,reactive_power_kvar,voltage_v,current_a,sub_metering_1,sub_metering_2,sub_metering_3,timestamp_utc,hour,day_of_week,month,day_of_month,is_weekend,quarter,active_power_kw_lag_1,active_power_kw_lag_24,active_power_kw_lag_168,active_power_kw_rolling_mean_3,active_power_kw_rolling_std_3,active_power_kw_rolling_mean_24,active_power_kw_rolling_std_24,active_power_kw_rolling_mean_168,active_power_kw_rolling_std_168,split
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00+00:00,17,5,12,16,1,4,NaN,NaN,NaN,4.216000,NaN,4.216000,NaN,4.216000,NaN,train
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00+00:00,17,5,12,16,1,4,4.216,NaN,NaN,4.788000,0.808930,4.788000,0.808930,4.788000,0.808930,train
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00+00:00,17,5,12,16,1,4,5.360,NaN,NaN,4.983333,0.664567,4.983333,0.664567,4.983333,0.664567,train
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00+00:00,17,5,12,16,1,4,5.374,NaN,NaN,5.374000,0.014000,5.084500,0.579113,5.084500,0.579113,train
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00+00:00,17,5,12,16,1,4,5.388,NaN,NaN,4.809333,0.990180,4.800800,0.808676,4.800800,0.808676,train



Previewing: electricity_load_diagrams_clean_raw.parquet


,unnamed_0,mt_001,mt_002,mt_003,mt_004,mt_005,mt_006,mt_007,mt_008,mt_009,mt_010,mt_011,mt_012,mt_013,mt_014,mt_015,mt_016,mt_017,mt_018,mt_019,mt_020,mt_021,mt_022,mt_023,mt_024,mt_025,mt_026,mt_027,mt_028,mt_029,mt_030,mt_031,mt_032,mt_033,mt_034,mt_035,mt_036,mt_037,mt_038,mt_039,mt_040,mt_041,mt_042,mt_043,mt_044,mt_045,mt_046,mt_047,mt_048,mt_049,mt_050,mt_051,mt_052,mt_053,mt_054,mt_055,mt_056,mt_057,mt_058,mt_059,mt_060,mt_061,mt_062,mt_063,mt_064,mt_065,mt_066,mt_067,mt_068,mt_069,mt_070,mt_071,mt_072,mt_073,mt_074,mt_075,mt_076,mt_077,mt_078,mt_079,mt_080,mt_081,mt_082,mt_083,mt_084,mt_085,mt_086,mt_087,mt_088,mt_089,mt_090,mt_091,mt_092,mt_093,mt_094,mt_095,mt_096,mt_097,mt_098,mt_099,...,mt_272,mt_273,mt_274,mt_275,mt_276,mt_277,mt_278,mt_279,mt_280,mt_281,mt_282,mt_283,mt_284,mt_285,mt_286,mt_287,mt_288,mt_289,mt_290,mt_291,mt_292,mt_293,mt_294,mt_295,mt_296,mt_297,mt_298,mt_299,mt_300,mt_301,mt_302,mt_303,mt_304,mt_305,mt_306,mt_307,mt_308,mt_309,mt_310,mt_311,mt_312,mt_313,mt_314,mt_315,mt_316,mt_317,mt_318,mt_319,mt_320,mt_321,mt_322,mt_323,mt_324,mt_325,mt_326,mt_327,mt_328,mt_329,mt_330,mt_331,mt_332,mt_333,mt_334,mt_335,mt_336,mt_337,mt_338,mt_339,mt_340,mt_341,mt_342,mt_343,mt_344,mt_345,mt_346,mt_347,mt_348,mt_349,mt_350,mt_351,mt_352,mt_353,mt_354,mt_355,mt_356,mt_357,mt_358,mt_359,mt_360,mt_361,mt_362,mt_363,mt_364,mt_365,mt_366,mt_367,mt_368,mt_369,mt_370,split
0,2011-01-01 00:15:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,"107,06932052162","261,261261261261","181,818181818182","70,9570041608876","922,277227722772","53,8353863381859","49,3966151582046","10278,3333333333",0,"71,8457802064359","52,293956043956","151,675749318801","45,187969924812","53,7976782752902","63,2739804709937","81,998031496063","40,7420924574209",0,"318,15331010453","83,695652173913","160,564334085779","74,2053930005737","393,885135135135","156,536144578313","47,8844862323707","77,7155172413793","860,882352941176","260,26936026936","100,027198549411","74,4064093226511","76,7471819645733","95,4868154158215","33,5876993166287",0,"36,7447784004075","364,769230769231",0,"152,728731942215","333,183856502242","100,250240615977","203,90625","104,485006518905","814,606741573034","72,6169749727965","167,829181494662","269,818731117825","93,3640081799591","47,336089781453","62,5195094760312","56,7608861726509",0,"1003,78048780488","247,095070422535","177,559912854031","147,543640897756","264,11575562701","644,444444444444","135,514705882353","45,08125","129,62962962963",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0,0,0,0,0,0,0,0,train
1,2011-01-01 00:30:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,"105,696636925189","273,273273273273","179,60088691796","71,6504854368932","912,376237623762","50,4759238521837","50,8682855040471",10445,0,"76,1020036429873","51,6071428571429","148,950953678474","42,1754385964912","53,7976782752902","74,3595634692705","82,9822834645669","41,9586374695864",0,"332,12543554007","85,3260869565217","147,020316027088","73,0579460699943","373,614864864865","162,560240963855","50,5708529214238","70,8103448275862","872,647058823529","283,872053872054","99,1205802357208","75,8630735615441","71,1030595813205","81,2778904665314","34,1571753986333",0,"39,2969943963321","364,769230769231",0,"122,215088282504","328,699551569507","97,3628488931665","196,09375","111,003911342894","870,786516853933","68,79760609358","189,181494661922","251,691842900302","95,4192229038855","49,1080921441229","58,6176142697882","56,7608861726509",0,"967,195121951219","243,573943661972","177,559912854031","131,321695760598","264,11575562701","592,592592592593","129,632352941176","50,0875","119,212962962963",0,0,0,0,


01_data_preprocessing.ipynb completed.


In [2]:
# ==============================================
# PACKAGE ALL PREPROCESSING OUTPUTS FOR DOWNLOAD
# ==============================================

import shutil
from pathlib import Path

# Core folders from your notebook
BASE_DIR = Path(BASE_DIR)
PROCESSED_DIR = Path(PROCESSED_DIR)
MANIFEST_DIR = Path(MANIFEST_DIR)
LOG_DIR = Path(LOG_DIR)

EXPORT_DIR = BASE_DIR / "exports_for_download"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ZIP 1: all processed datasets
processed_zip = EXPORT_DIR / "01_processed_datasets.zip"
if processed_zip.exists():
    processed_zip.unlink()
shutil.make_archive(str(processed_zip.with_suffix('')), 'zip', root_dir=str(PROCESSED_DIR))

# ZIP 2: manifests and reports
manifests_zip = EXPORT_DIR / "01_manifests_and_reports.zip"
if manifests_zip.exists():
    manifests_zip.unlink()
temp_manifest_bundle = EXPORT_DIR / "temp_manifest_bundle"
if temp_manifest_bundle.exists():
    shutil.rmtree(temp_manifest_bundle)
temp_manifest_bundle.mkdir(parents=True, exist_ok=True)

# copy manifests
if MANIFEST_DIR.exists():
    shutil.copytree(MANIFEST_DIR, temp_manifest_bundle / "manifests", dirs_exist_ok=True)

# copy logs
if LOG_DIR.exists():
    shutil.copytree(LOG_DIR, temp_manifest_bundle / "logs", dirs_exist_ok=True)

# copy quality reports and feature manifests from processed folders
reports_dir = temp_manifest_bundle / "dataset_reports"
reports_dir.mkdir(parents=True, exist_ok=True)

for dataset_dir in PROCESSED_DIR.glob("dataset_*"):
    if dataset_dir.is_dir():
        dst = reports_dir / dataset_dir.name
        dst.mkdir(parents=True, exist_ok=True)
        for fname in ["quality_report.json", "feature_manifest.json", "graph_metadata.json", "rl_metadata.json"]:
            src = dataset_dir / fname
            if src.exists():
                shutil.copy2(src, dst / fname)

shutil.make_archive(str(manifests_zip.with_suffix('')), 'zip', root_dir=str(temp_manifest_bundle))
shutil.rmtree(temp_manifest_bundle, ignore_errors=True)

# ZIP 3: full bundle
full_bundle_zip = EXPORT_DIR / "01_full_preprocessing_bundle.zip"
if full_bundle_zip.exists():
    full_bundle_zip.unlink()
temp_full_bundle = EXPORT_DIR / "temp_full_bundle"
if temp_full_bundle.exists():
    shutil.rmtree(temp_full_bundle)
temp_full_bundle.mkdir(parents=True, exist_ok=True)

shutil.copytree(PROCESSED_DIR, temp_full_bundle / "processed", dirs_exist_ok=True)
if MANIFEST_DIR.exists():
    shutil.copytree(MANIFEST_DIR, temp_full_bundle / "manifests", dirs_exist_ok=True)
if LOG_DIR.exists():
    shutil.copytree(LOG_DIR, temp_full_bundle / "logs", dirs_exist_ok=True)

shutil.make_archive(str(full_bundle_zip.with_suffix('')), 'zip', root_dir=str(temp_full_bundle))
shutil.rmtree(temp_full_bundle, ignore_errors=True)

print("ZIP packages created successfully:")
print(processed_zip)
print(manifests_zip)
print(full_bundle_zip)

ZIP packages created successfully:
/content/drive/MyDrive/REAL DATASETS/exports_for_download/01_processed_datasets.zip
/content/drive/MyDrive/REAL DATASETS/exports_for_download/01_manifests_and_reports.zip
/content/drive/MyDrive/REAL DATASETS/exports_for_download/01_full_preprocessing_bundle.zip


In [3]:
# ==============================================
# DOWNLOAD ZIP FILES TO YOUR PC
# ==============================================

from google.colab import files

files.download(str(EXPORT_DIR / "01_processed_datasets.zip"))
files.download(str(EXPORT_DIR / "01_manifests_and_reports.zip"))
files.download(str(EXPORT_DIR / "01_full_preprocessing_bundle.zip"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# ==============================================
# SHOW EXPORT FILES
# ==============================================

for p in sorted(EXPORT_DIR.glob("*.zip")):
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"{p.name}  |  {size_mb:.2f} MB")

01_full_preprocessing_bundle.zip  |  175.34 MB
01_manifests_and_reports.zip  |  0.04 MB
01_processed_datasets.zip  |  175.33 MB
